In [2]:
import asyncio
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor
import nest_asyncio
nest_asyncio.apply()  # For Jupyter/Colab
import numpy as np
from dotenv import load_dotenv
import os
import pandas as pd
from tqdm.asyncio import tqdm
import time
import numpy as np
import json

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

# Create multiple model instances for parallel calls
def create_gemini_model():
    """Create Gemini model with JSON response schema"""
    import google.generativeai as genai
    
    generation_config = {
        "temperature": 0.1,  # Very low for lite model consistency
        "response_mime_type": "application/json",
        "response_schema": {
            "type": "object",
            "properties": {
                "correctness": {
                    "type": "number",
                    "description": "Score 0-10"
                },
                "reasoning_quality": {
                    "type": "number", 
                    "description": "Score 0-10"
                },
                "difficulty": {
                    "type": "string",
                    "enum": ["EASY", "MEDIUM", "HARD"]
                }
            },
            "required": ["correctness", "reasoning_quality", "difficulty"]
        }
    }
    
    model = genai.GenerativeModel(
        model_name='gemini-2.0-flash',
        generation_config=generation_config
    )
    
    return model

# from google.cloud import aiplatform
# import vertexai
# from vertexai.generative_models import GenerativeModel

# # Initialize Vertex AI
# vertexai.init(project="kaggle-tunix", location="us-central1")

# def create_gemini_model():
#     """Use Vertex AI with DSQ - no daily limits"""
#     model = GenerativeModel(
#         "gemini-2.5-flash",
#             generation_config = {
#         "temperature": 0.2,
#         "response_mime_type": "application/json",
#         "response_schema": {
#             "type": "OBJECT",
#             "properties": {
#                 "correctness": {
#                     "type": "NUMBER",
#                     "description": "Score 0-10"
#                 },
#                 "reasoning_quality": {
#                     "type": "NUMBER",
#                     "description": "Score 0-10"
#                 },
#                 "difficulty": {
#                     "type": "STRING",
#                     "enum": ["EASY", "MEDIUM", "HARD"]
#                 }
#             },
#             "required": ["correctness", "reasoning_quality", "difficulty"]
#         }
#     }
#     )
#     return model

In [ ]:
# !gcloud auth login
# # !gcloud config set project kaggle-tunix

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=0rfmtthYTHIsSEb0TLLFikp5ySOO1Y&access_type=offline&code_challenge=h9Wq4mldF09nFzLOnI779Pe6JVRz6yQR_tVJ4Ri-gzw&code_challenge_method=S256


You are now logged in as [hazardscarn10@gmail.com].
Your current project is [kaggle-tunix].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [23]:


def parse_scores(result_text):
    """Parse JSON response"""
    try:
        scores = json.loads(result_text)
        
        required = ['correctness', 'reasoning_quality', 'difficulty']
        if not all(key in scores for key in required):
            print(f"Missing fields: {scores}")
            return None
            
        return scores
        
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        print(f"Raw: {result_text[:200]}")
        return None


def evaluate_teacher_response(input_text, model_reasoning, model_response, ground_truth, domain, model):
    '''Evaluate teacher-generated samples - returns JSON only'''
    
    prompt = f'''You are grading AI training data quality.This data will be used to train GEMMA3 1B model with reasoning added.
    For each question below, the reasoning and response have been created by another model and we will use this to tech Gemma model how to reason and make correct answer.
    Inorder to make sure we are training the Gemma model with best data, we need you to evaluate this training reasoning and response the teacher model has created to make sure we teach Gemma model the 
    with top quality samples. Return only JSON.
    

DOMAIN: {domain}

QUESTION: {input_text}

GROUND TRUTH: {ground_truth}

AI REASONING: {model_reasoning}

MODEL ANSWER: {model_response}

SCORING GUIDELINES:

1.ANSWER CORRECTNESS (0-10) - Rate MODEL ANSWER vs GROUND TRUTH:
    For verifiable tasks:
    - 0-2: Wrong answer
    - 3-5: Partially correct, missing key elements
    - 6-8: Mostly correct, minor errors
    - 9-10: Perfectly matches ground truth
    - CRITICAL INSTRUCTION FOR EXPRESSIONS:
        - If GROUND TRUTH is a mathematical expression like "(24.7+38.8)/2" or "(500-3373)/3373"
        - And MODEL ANSWER is the evaluated result like "31.75" or "-85.2%"  
        - These are SEMANTICALLY EQUIVALENT → Score 9-10 for correctness
        - Do NOT penalize for showing work vs showing formula

    For subjective tasks (writing, summarization):
    - 0-2: Fails task requirements
    - 3-5: Meets basic requirements, low quality
    - 6-8: Good quality, fulfills task well
    - 9-10: Excellent quality


2) REASONING QUALITY (0-10): Is AI REASONING logically sound?

Score based on LOGIC, not style:
- 0-3: Contains mathematical errors, wrong formulas, or contradictions
- 4-6: Logic is correct but explanation is unclear or skips steps
- 7-8: Correct logic, all steps shown, minor verbosity
- 9-10: Correct logic, clear explanation, optimal presentation
- Note: Score low if reasoning contains errors even if final answer is correct.

DIFFICULTY:
- EASY: Direct facts/formulas, 1-3 steps
- MEDIUM: Multiple concepts, 4-6 steps, option elimination
- HARD: Complex multi-stage, 7+ steps, novel decomposition
Return ONLY a JSON object with no additional text.
Return JSON format:
{{
  "correctness": 8,
  "reasoning_quality": 7,
  "difficulty": "MEDIUM"
}}'''
    
    try:
        response = model.generate_content(prompt)
        return parse_scores(response.text)
    except Exception as e:
        print(f"Error evaluating: {e}")
        return None


class AsyncGeminiEvaluator:
    def __init__(self, max_concurrent=30, max_retries=3):
        """
        max_concurrent: 30 is safer for Gemini rate limits
        max_retries: Number of retry attempts
        """
        self.max_concurrent = max_concurrent
        self.max_retries = max_retries
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.executor = ThreadPoolExecutor(max_workers=max_concurrent)
        
    # async def evaluate_single_async(self, idx, row):
    #     """Async wrapper with retry logic"""
    #     async with self.semaphore:
    #         for attempt in range(self.max_retries):
    #             try:
    #                 loop = asyncio.get_event_loop()
                    
    #                 result = await loop.run_in_executor(
    #                     self.executor,
    #                     self._evaluate_blocking,
    #                     idx, row
    #                 )
    #                 return result
                    
    #             except Exception as e:
    #                 if attempt == self.max_retries - 1:
    #                     print(f"\nFailed row {idx} after {self.max_retries} attempts: {e}")
    #                     return idx, None
                    
    #                 # Exponential backoff
    #                 wait_time = 2 ** attempt
    #                 await asyncio.sleep(wait_time)
    
    async def evaluate_single_async(self, idx, row):
        """Async wrapper with retry logic"""
        async with self.semaphore:
            for attempt in range(self.max_retries):
                try:
                    loop = asyncio.get_event_loop()
                    
                    result = await loop.run_in_executor(
                        self.executor,
                        self._evaluate_blocking,
                        idx, row
                    )
                    return result
                    
                except Exception as e:
                    if "429" in str(e) or "Resource exhausted" in str(e):
                        # For rate limits, wait longer
                        wait_time = 60  # Wait 1 minute on rate limit
                        print(f"\n429 error on row {idx}, waiting {wait_time}s...")
                        await asyncio.sleep(wait_time)
                    elif attempt == self.max_retries - 1:
                        print(f"\nRow {idx} failed after {self.max_retries} attempts")
                        return idx, None
                    else:
                        # Regular retry backoff
                        wait_time = 2 ** attempt
                        await asyncio.sleep(wait_time)
        
        return idx, None
    
    def _evaluate_blocking(self, idx, row):
        """Blocking evaluation - creates fresh model per call"""
        try:
            # Create model with JSON schema per call
            model = create_gemini_model()
            
            scores = evaluate_teacher_response(
                input_text=row['input'],
                model_reasoning=row['cleaned_reasoning'],
                model_response=row['response'],
                ground_truth=row['ground_truth'],
                domain=row.get('domain', 'general'),
                model=model
            )
            
            return idx, scores
            
        except Exception as e:
            raise
    
    async def evaluate_all(self, eval_df):
        """Evaluate with real-time progress"""
        tasks = [
            self.evaluate_single_async(idx, row) 
            for idx, row in eval_df.iterrows()
        ]
        
        print(f"\nEvaluating {len(tasks)} responses ({self.max_concurrent} concurrent)...")
        
        results = []
        for coro in tqdm.as_completed(tasks, total=len(tasks)):
            result = await coro
            results.append(result)
        
        self.executor.shutdown(wait=True)
        return results


def evaluate_model(eval_df,max_concurrent=30):
    """Main evaluation with error tracking"""
    start_time = time.time()
    
    eval_df = eval_df.copy()
    evaluator = AsyncGeminiEvaluator(max_concurrent=max_concurrent)
    
    results = asyncio.run(evaluator.evaluate_all(eval_df))
    
    # Process results
    correctness_scores = []
    reasoning_scores = []
    difficulties = []
    failed_indices = []
    
    for idx, scores in results:
        if scores:
            correctness_scores.append(scores.get('correctness', 0))
            reasoning_scores.append(scores.get('reasoning_quality', 0))
            difficulties.append(scores.get('difficulty', 'UNKNOWN'))
        else:
            failed_indices.append(idx)
            correctness_scores.append(0)
            reasoning_scores.append(0)
            difficulties.append('UNKNOWN')
    
    eval_df['correctness'] = correctness_scores
    eval_df['reasoning_quality'] = reasoning_scores
    eval_df['difficulty'] = difficulties
    
    elapsed = time.time() - start_time
    
    print(f"\n{'='*60}")
    print(f"Evaluation Complete: {elapsed:.1f}s ({len(eval_df)/elapsed:.1f} eval/sec)")
    print(f"{'='*60}")
    print(f"Success: {len(eval_df) - len(failed_indices)}/{len(eval_df)}")
    print(f"Failed: {len(failed_indices)}")
    
    # Statistics
    valid_correctness = [s for s in correctness_scores if s > 0]
    valid_reasoning = [s for s in reasoning_scores if s > 0]
    
    if valid_correctness:
        print(f"\nQuality Metrics:")
        print(f"  Correctness:       {np.mean(valid_correctness):.2f}/10 (±{np.std(valid_correctness):.2f})")
        print(f"  Reasoning Quality: {np.mean(valid_reasoning):.2f}/10 (±{np.std(valid_reasoning):.2f})")
        
        # Difficulty distribution
        difficulty_counts = eval_df[eval_df['difficulty'] != 'UNKNOWN']['difficulty'].value_counts()
        print(f"\nDifficulty Distribution:")
        for diff in ['EASY', 'MEDIUM', 'HARD']:
            count = difficulty_counts.get(diff, 0)
            pct = count / len(valid_correctness) * 100 if valid_correctness else 0
            print(f"  {diff}: {count} ({pct:.1f}%)")
    
    return eval_df, failed_indices

def evaluate_in_batches(df, batch_size=1000, delay_between_batches=30,max_concurrent=10):
    """Evaluate in batches with cooling periods"""
    results = []
    
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]
        print(f"\nProcessing batch {i//batch_size + 1}/{len(df)//batch_size + 1}")
        
        eval_batch, failed = evaluate_model(batch, max_concurrent=max_concurrent)
        results.append(eval_batch)
        
        if i + batch_size < len(df):
            print(f"Waiting {delay_between_batches}s before next batch...")
            time.sleep(delay_between_batches)
    
    return pd.concat(results)


def filter_quality_samples(eval_df, target_count=20000):
    """Filter top quality samples"""
    
    print(f"\nFiltering from {len(eval_df)} samples...")
    
    # Quality filters
    filtered = eval_df[
        (eval_df['correctness'] >= 7) &
        (eval_df['reasoning_quality'] >= 6) &
        (eval_df['difficulty'] != 'UNKNOWN')
    ].copy()
    
    print(f"After quality filters: {len(filtered)}/{len(eval_df)}")
    
    if len(filtered) == 0:
        print("WARNING: No samples passed filters!")
        return filtered
    
    # Composite score
    filtered['composite_score'] = (
        0.5 * filtered['correctness'] + 
        0.5 * filtered['reasoning_quality']
    )
    
    # Take top N
    if len(filtered) > target_count:
        filtered = filtered.nlargest(target_count, 'composite_score')
        print(f"Selected top {target_count} by composite score")
    
    # Statistics
    print(f"\nFinal Dataset:")
    print(f"  Samples: {len(filtered)}")
    print(f"  Avg Correctness: {filtered['correctness'].mean():.2f}/10")
    print(f"  Avg Reasoning: {filtered['reasoning_quality'].mean():.2f}/10")
    
    difficulty_dist = filtered['difficulty'].value_counts()
    print(f"\nDifficulty:")
    for diff, count in difficulty_dist.items():
        print(f"  {diff}: {count} ({count/len(filtered)*100:.1f}%)")
    
    return filtered

In [39]:
train_df= pd.read_parquet('../data//final_data//custom_df_train_sft.parquet')
train_df=train_df[['input','ground_truth','cleaned_reasoning','response','domain','uid']]

In [24]:
# Process all 51K in batches
eval_df = evaluate_in_batches(train_df, batch_size=1500, delay_between_batches=30,max_concurrent=20)



Processing batch 1/35

Evaluating 1500 responses (20 concurrent)...


  0%|          | 0/1500 [00:00<?, ?it/s]

100%|██████████| 1500/1500 [00:41<00:00, 36.05it/s]



Evaluation Complete: 41.7s (36.0 eval/sec)
Success: 1500/1500
Failed: 0

Quality Metrics:
  Correctness:       9.48/10 (±0.99)
  Reasoning Quality: 8.46/10 (±1.00)

Difficulty Distribution:
  EASY: 429 (29.0%)
  MEDIUM: 1070 (72.2%)
  HARD: 1 (0.1%)
Waiting 30s before next batch...

Processing batch 2/35

Evaluating 1500 responses (20 concurrent)...


100%|██████████| 1500/1500 [00:41<00:00, 35.87it/s]


...


Evaluation Complete: 42.6s (35.2 eval/sec)
Success: 1500/1500
Failed: 0

Quality Metrics:
  Correctness:       9.48/10 (±0.89)
  Reasoning Quality: 8.48/10 (±0.95)

Difficulty Distribution:
  EASY: 412 (27.7%)
  MEDIUM: 1085 (72.8%)
  HARD: 3 (0.2%)
Waiting 30s before next batch...

Processing batch 34/35

Evaluating 1500 responses (20 concurrent)...


100%|██████████| 1500/1500 [00:43<00:00, 34.26it/s]



Evaluation Complete: 43.8s (34.2 eval/sec)
Success: 1500/1500
Failed: 0

Quality Metrics:
  Correctness:       9.41/10 (±1.06)
  Reasoning Quality: 8.42/10 (±0.99)

Difficulty Distribution:
  EASY: 418 (28.0%)
  MEDIUM: 1080 (72.4%)
  HARD: 2 (0.1%)
Waiting 30s before next batch...

Processing batch 35/35

Evaluating 614 responses (20 concurrent)...


100%|██████████| 614/614 [00:18<00:00, 33.77it/s]


Evaluation Complete: 18.2s (33.7 eval/sec)
Success: 614/614
Failed: 0

Quality Metrics:
  Correctness:       9.48/10 (±1.07)
  Reasoning Quality: 8.53/10 (±1.00)

Difficulty Distribution:
  EASY: 187 (30.6%)
  MEDIUM: 426 (69.7%)
  HARD: 1 (0.2%)


In [27]:
eval_df.shape

(51614, 8)

In [36]:
# Check how many have 0 scores
zero_scores = eval_df[
    (eval_df['correctness'] <= 3) | 
    (eval_df['reasoning_quality'] <= 3)
]

print(f"Samples with low scores: {len(zero_scores)}/{eval_df.shape[0]} ({np.round(len(zero_scores)*100/eval_df.shape[0],2)}%)")

# Check if they're mostly expression-based ground truths
expression_pattern = zero_scores['ground_truth'].str.contains(r'[\+\-\*/\(\)]', regex=True, na=False)
print(f"Of those, {expression_pattern.sum()} have expression-based ground truth")

Samples with low scores: 841/51614 (1.63%)
Of those, 108 have expression-based ground truth


In [ ]:
# eval_df2=pd.merge(eval_df,train_df[['input','uid','ground_truth','domain']],on=['input','ground_truth','domain'],how='inner')
# eval_df2.shape

(57430, 9)

In [37]:
eval_df.to_parquet('../data//final_data//custom_df_train_sft_quality_tested.parquet')

In [38]:
eval_df.sample(20)

,input,ground_truth,cleaned_reasoning,response,domain,correctness,reasoning_quality,difficulty
34054,Context:\nPeter F. Paul: Peter Franklin Paul (...,Peter F. Paul,This is a commonsense reasoning task.\n\nWe ne...,The individual was **Peter F. Paul** – a forme...,commonsense_reasoning,10.0,10.0,MEDIUM
32324,"Context:\nVic Morrow: Victor ""Vic"" Morrow (Feb...",Marmaduke,This is a commonsense reasoning task.\n\nWe ne...,The director is **Tom Dey**. In addition to *S...,commonsense_reasoning,9.0,9.0,MEDIUM
34480,Context:\nIndica (Ctesias): Indica (Greek: Ἰνδ...,Dioscorea,This is a commonsense reasoning task.\n\nWe ne...,The plant **Dioscorea** was named after a Gree...,commonsense_reasoning,10.0,8.0,EASY
40652,"Table :\n"""","""",""Year ended March 31, "",""""\n"""",...",31.1 - 14.1,This is a numerical reasoning task.\n\nWe need...,"**Extracted values**\n\n- U.S. state taxes, ne...",numerical_reasoning,9.0,8.0,MEDIUM
50639,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello. How can I help you toda...,This is a conversational task.\n\nThe user wan...,User: Hi \nAI: Hello! How can I help you toda...,conversational,9.0,8.0,MEDIUM
29962,Question: Could Bart Simpson have owned comics...,True,This is a commonsense reasoning task.\n\nWe ne...,**Short answer:** Yes—both in the real‑world t...,commonsense_reasoning,10.0,10.0,EASY
1348,A single kabob stick has 4 cubes of beef. A sl...,50,This is a mathematical reasoning task.\n\nWe n...,Each kabob stick needs 4 cubes of beef.\n\n**C...,math,10.0,9.0,MEDIUM
635,A baker has 10 cheesecakes on the display whil...,18,This is a mathematical reasoning task.\n\nWe n...,The baker started with \n\n- Display: 10 chee...,math,9.0,8.0,MEDIUM
43573,hr solutions . <table class='wikitable'><tr><t...,31.0,This is a financial reasoning task.\n\nThe use...,The operating income in 2010 was **$234 millio...,financial_reasoning,8.0,7.0,MEDIUM
2069,Page collects fancy shoes and has 80 pairs in ...,62,This is a mathematical reasoning task.\n\nWe n...,**Step‑by‑step**\n\n1. **Start:** 80 pairs of ...,math,10.0,9.0,MEDIUM
